# Лабораторная работа № 2. 128-битные числа Big Endian

**Выполнил** Зелёный Артём Сергеевич  
**Группа** М8О-209СВ-25  
**Вариант** 10

**Цель.** Реализовать `OR`, `AND`, `NOT`, `XOR` и побитовое сложение для 128-битных беззнаковых чисел.

**Источники.** «Лаб - Большие числа FULL.pptx», слайд 12, и «BigNumber — шаблон.txt».

## 1. Логические операции и сумматор

`NAND(a, b)` равен нулю только при двух единицах. Соединяя несколько таких операций, получаем остальные логические операции. Полусумматор возвращает пару `(перенос, сумма)`. Полный сумматор складывает два бита и входящий перенос с помощью двух полусумматоров.

In [1]:
def nand(a, b):
    return int(not (a and b))

def bit_not(a):
    return nand(a, a)

def bit_and(a, b):
    return bit_not(nand(a, b))

def bit_or(a, b):
    return nand(bit_not(a), bit_not(b))

def bit_xor(a, b):
    shared = nand(a, b)
    return nand(nand(a, shared), nand(b, shared))

def half_add(a, b):
    return bit_and(a, b), bit_xor(a, b)

def full_add(a, b, carry):
    carry_ab, partial = half_add(a, b)
    carry_partial, result = half_add(partial, carry)
    return bit_or(carry_ab, carry_partial), result

## 2. Представление и сложение

Нулевой индекс хранит старший бит. Короткий набор битов дополняется нулями слева. При сложении обход идёт справа налево от бита 127 к биту 0. Результат хранит младшие 128 бит, а перенос за старший разряд записывается в `overflow`. Преобразование в `int` используется для вывода и проверки.

In [2]:
class BigNumber:
    WIDTH = 128

    def __init__(self, bits=()):
        bits = tuple(bits)
        if len(bits) > self.WIDTH or any(bit not in (0, 1) for bit in bits):
            raise ValueError('Ожидалось до 128 двоичных разрядов')
        self.bits = (0,) * (self.WIDTH - len(bits)) + tuple(map(int, bits))
        self.overflow = False

    @classmethod
    def from_int(cls, value):
        if not 0 <= value < 1 << cls.WIDTH:
            raise ValueError('Число выходит за диапазон 128-битного беззнакового типа')
        return cls(map(int, f'{value:b}'))

    def __str__(self):
        return ''.join(map(str, self.bits))

    def __int__(self):
        return int(str(self), 2)

    def __and__(self, other):
        if not isinstance(other, BigNumber):
            return NotImplemented
        return BigNumber(bit_and(a, b) for a, b in zip(self.bits, other.bits))

    def __or__(self, other):
        if not isinstance(other, BigNumber):
            return NotImplemented
        return BigNumber(bit_or(a, b) for a, b in zip(self.bits, other.bits))

    def __xor__(self, other):
        if not isinstance(other, BigNumber):
            return NotImplemented
        return BigNumber(bit_xor(a, b) for a, b in zip(self.bits, other.bits))

    def __invert__(self):
        return BigNumber(bit_not(bit) for bit in self.bits)

    def __add__(self, other):
        if not isinstance(other, BigNumber):
            return NotImplemented
        result_bits = [0] * self.WIDTH
        carry = 0
        for index in range(self.WIDTH - 1, -1, -1):
            carry, result_bits[index] = full_add(self.bits[index], other.bits[index], carry)
        result = BigNumber(result_bits)
        result.overflow = bool(carry)
        return result

## 3. Проверка таблиц истинности и 128-битных результатов

Проверяем все восемь наборов входов полного сумматора, порядок битов, перенос между разрядами и переполнение. Сравниваем арифметику с целыми числами Python.

In [3]:
from itertools import product
from random import Random

print('a b c_in | c_out sum')
for a, b, carry in product((0, 1), repeat=3):
    result = full_add(a, b, carry)
    assert result == divmod(a + b + carry, 2)
    assert (bit_and(a, b), bit_or(a, b), bit_xor(a, b), bit_not(a)) == (a & b, a | b, a ^ b, 1 - a)
    print(a, b, carry, '|', *result)

mask = (1 << BigNumber.WIDTH) - 1
assert str(BigNumber.from_int(1)) == '0' * 127 + '1'
assert str(BigNumber.from_int(1 << 127)) == '1' + '0' * 127
rng = Random(10)
pairs = [(0, 0), (5, 3), (255, 1), (mask, 0), (mask, 1), (mask, mask)]
pairs += [(rng.getrandbits(128), rng.getrandbits(128)) for _ in range(32)]

for x, y in pairs:
    a, b = BigNumber.from_int(x), BigNumber.from_int(y)
    assert int(a & b) == x & y
    assert int(a | b) == x | y
    assert int(a ^ b) == x ^ y
    assert int(~a) == (~x & mask)
    total = a + b
    assert int(total) == (x + y) & mask
    assert total.overflow == (x + y > mask)

overflowing = BigNumber.from_int(mask) + BigNumber.from_int(1)
print(f'Проверено пар 128-битных чисел: {len(pairs)}')
print(f'(2¹²⁸ − 1) + 1 даёт {int(overflowing)}, overflow = {overflowing.overflow}')

a b c_in | c_out sum
0 0 0 | 0 0
0 0 1 | 0 1
0 1 0 | 0 1
0 1 1 | 1 0
1 0 0 | 0 1
1 0 1 | 1 0
1 1 0 | 1 0
1 1 1 | 1 1
Проверено пар 128-битных чисел: 38
(2¹²⁸ − 1) + 1 даёт 0, overflow = True


## Вывод

Реализованы четыре логические операции и сложение с переносом. Для разрядности $w$ каждая операция проходит по $w$ битам и имеет сложность $O(w)$.